# Imports

In [ ]:
from data.custom_dataset import CustomImageFolder # needed to create pytorch dataloader

from training.train_loop import train_loop # main function with wich you train a model
from training.eval_model import eval_model # function wich allows you to assess the model on a validation dataset
from training.accuracy_fn import accuracy_fn # function for assess accuracy

from models import models # all 3 architectures

# For data preparation
import zipfile
import shutil
import glob
import torchvision
from torchvision.transforms import v2
from torch.utils.data import DataLoader

# TPU initialization
import torch_xla
import torch_xla.core.xla_model as xm
import torch_xla.distributed.parallel_loader as pl
import torch_xla.utils.utils as xu
import torch_xla.distributed.xla_multiprocessing as xmp
from torch_xla.distributed.parallel_loader import MpDeviceLoader
device = xm.xla_device()

# Data preparation

In [ ]:
# Unpack .zip files train.zip and test.zip which contain train and test data respectively
# Each file has the same structure as follows:
# train.zip/
# └── train_knots/
#     └── 0 <--- number of crossings
#         └── 0.png <── sample images
#         └── 1.png <──┘
#         └── ...
#     └── 3
#         └── 0.png
#         └── 1.png
#         └── ...
#     └── ...

!unpack train.zip -d './'
!unpack test.zip -d './'

In [ ]:
# Transforms used to preprocess the data (described in the paper)
data_transform = v2.Compose([v2.Grayscale(num_output_channels=1),
                             v2.ToImage(),
                             v2.ToDtype(torch.float32, scale=True),
                             v2.functional.invert,
                             ])

train_data = CustomImageFolder(root='./train_knots',
                         transform=data_transform)
test_data = CustomImageFolder(root='./test_knots',
                         transform=data_transform)

In [ ]:
# Setup the batch size hyperparameter
BATCH_SIZE = 128

# Turn datasets into iterables (batches)
train_dataloader = DataLoader(train_data,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

test_dataloader = DataLoader(test_data,
                             num_workers=2,
                             pin_memory=True)

train_dataloader = MpDeviceLoader(train_dataloader, device)
test_dataloader = MpDeviceLoader(test_dataloader, device)

# Training a model

Let's train a transformer (you can train other models as well, but remember to change parameters described in the paper)

In [ ]:
model_cvt = models.KnotsModelCvT(
    Ns=(2, 3, 9), # parameters are described in the paper
    params=(
        (32, 32, 1, 32, 8, 256, 32),
        (8, 4, 32, 64, 8, 256, 64),
        (3, 1, 64, 128, 8, 256, 128),
    )
).to(device)

In [ ]:
loss_fn = nn.MSELoss() # loss function

optimizer = torch.optim.AdamW(params=model_cvt.parameters(), # optimizer
                              lr=0.003)

START_EPOCH = 30 # parameter used to prevent early scheduling

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, # scheduler
                                                      mode='min',
                                                      factor=0.7,
                                                      patience=1,
                                                      threshold=0.001,
                                                      cooldown=7)

In [ ]:
EPOCHS = 70 # number of epochs model'll train

# Different statistics; used to plot time usage/learning rate's changes/confusion matrices etc.
y_pred_train, y_target_train, train_losses, train_accuracies = torch.Tensor(), torch.Tensor(), [], []
y_pred_test, y_target_test, test_losses, test_accuracies = torch.Tensor(), torch.Tensor(), [], []
times = []
lrs = []

# Main function
train_loop(model_cvt, START_EPOCH, train_dataloader, test_dataloader, loss_fn, optimizer, accuracy_fn, EPOCHS, scheduler)

# Model evaluation
eval_model(model_cvt, test_dataloader, loss_fn, accuracy_fn)